# settings

In [1]:
# 필요한 라이브러리 로드
import requests 
import os
from dotenv import load_dotenv
import pandas as pd
import geopandas as gpd
import re
import time

In [2]:
# env 파일 로드
load_dotenv()

True

## 함수

In [3]:
# kakao API로 주소, 위도, 경도 가져오는 함수
def add_kakao_geocode(df, kakao_key, address_col="addr", out_col = "rd_addr", sleep_sec=0.03, max_retry=5):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    out = df.copy()

    for c in [out_col, "lat", "lng"]:
        if c not in out.columns:
            out[c] = pd.NA

    session = requests.Session()
    session.headers.update({"Authorization": f"KakaoAK {kakao_key}"})

    # 중복 주소 1회만 호출
    uniq_addrs = (
        out[address_col]
        .dropna()
        .astype(str)
        .str.strip()
    )
    uniq_addrs = uniq_addrs[uniq_addrs != ""].drop_duplicates()

    cache = {}  # {addr: (road_address, lat, lng)}

    for add in uniq_addrs:
        params = {"query": add}
        result = None

        for i in range(max_retry):
            try:
                r = session.get(url, params=params, timeout=10)

                if r.status_code == 429:  # rate limit
                    time.sleep((i + 1) * 0.8)
                    continue

                r.raise_for_status()
                result = r.json()
                break
            except requests.exceptions.RequestException:
                time.sleep((i + 1) * 0.8)

        if not result:
            cache[add] = (pd.NA, pd.NA, pd.NA)
            continue

        docs = result.get("documents", [])
        if not docs:
            cache[add] = (pd.NA, pd.NA, pd.NA)
            continue

        road = docs[0].get("road_address")
        if road:
            cache[add] = (road.get("address_name"), road.get("y"), road.get("x"))
        else:
            addr = docs[0].get("address") or {}
            cache[add] = (addr.get("address_name"), addr.get("y"), addr.get("x"))

        time.sleep(sleep_sec)

    mapped = out[address_col].astype(str).str.strip().map(cache)
    out[[out_col, "lat", "lng"]] = mapped.apply(pd.Series)

    out["lat"] = pd.to_numeric(out["lat"], errors="coerce")
    out["lng"] = pd.to_numeric(out["lng"], errors="coerce")

    return out

In [4]:
# 상권 및 행정동 코드 추출
def spatial_join_code(target_df, target_gdf, shp_path, shp_col, out_col):
    """
    target_gdf: 이미 만들어진 GeoDataFrame (EPSG:4326)
    shp_path: shp 또는 zip 경로
    shp_col: shp에서 가져올 컬럼명
    out_col: cafe_df에 붙일 컬럼명
    """
    # shp 로드 & 좌표계 통일
    gdf = gpd.read_file(shp_path)
    crs_5181_gdf = target_gdf.to_crs("EPSG:5181") 
    '''
    폴리곤(상권)은 꼭짓점이 많아서 변환 시 오차가 누적될 수 있으므로 shp(5181) 기준으로 맞춤
    '''
    
    # 공간조인
    joined = gpd.sjoin(
        crs_5181_gdf,
        gdf[[shp_col, "geometry"]],
        how="left",
        predicate="within"
    )

    # 결과 붙이기
    target_df = target_df.copy()
    target_df.loc[joined.index, out_col] = joined[shp_col].values

    return target_df

# 지식산업센터

## 데이터 불러오기

In [132]:
apikey  = os.getenv("MJ_APIKEY")     # 인증키
start_index = "1"    # 요청시작위치
end_index   = "300"  # 요청종료위치

url = f"http://openapi.seoul.go.kr:8088/{apikey}/json/InfoBizcenterApply/{start_index}/{end_index}/"

response = requests.get(url)
print(response.content.decode('utf-8'))

{"InfoBizcenterApply":{"list_total_count":213,"RESULT":{"CODE":"INFO-000","MESSAGE":"정상 처리되었습니다"},"row":[{"IDX":208.0,"FAC_NM":"신내테크노타운","ADDRESS":"봉화산로 123(상봉동)","TELEPHONE":"02)3423-0057","APPROVAL_DATE":"96.5.28","COMPLE_DATE":"99.10.18"},{"IDX":136.0,"FAC_NM":"월곡동아파트형공장","ADDRESS":"하월곡동 96-113","TELEPHONE":"02-090-8083","APPROVAL_DATE":"33289","COMPLE_DATE":"1990.12.31"},{"IDX":135.0,"FAC_NM":"월곡동아파트형공장","ADDRESS":"하월곡동 96-113","TELEPHONE":"02-912-1034","APPROVAL_DATE":"33289","COMPLE_DATE":"1990.12.31"},{"IDX":140.0,"FAC_NM":"문정아르페온 1차 지식산업센터","ADDRESS":"서울특별시 송파구 문정동 640\n(문정도시개발구역 5-1BL)","TELEPHONE":"031-706-8577","APPROVAL_DATE":"2016.06.27","COMPLE_DATE":"2016.06.27"},{"IDX":139.0,"FAC_NM":"문정 대명벨리온","ADDRESS":"서울특별시 송파구 문정동 641-4\n(문정도시개발구역 4-3BL)","TELEPHONE":"02-2222-7357","APPROVAL_DATE":"2016.03.23","COMPLE_DATE":"2016.03.23"},{"IDX":141.0,"FAC_NM":"문정 현대지식산업센터Ｉ-1","ADDRESS":"서울특별시 송파구 문정동 644 (문정 특별계획구역 6-1BL)","TELEPHONE":"02-3451-0100","APPROVAL_DATE":"2016.02.29","CO

In [133]:
data = response.json()
rows = data['InfoBizcenterApply']['row']
print(rows)

center_df = pd.DataFrame(rows)
center_df

[{'IDX': 208.0, 'FAC_NM': '신내테크노타운', 'ADDRESS': '봉화산로 123(상봉동)', 'TELEPHONE': '02)3423-0057', 'APPROVAL_DATE': '96.5.28', 'COMPLE_DATE': '99.10.18'}, {'IDX': 136.0, 'FAC_NM': '월곡동아파트형공장', 'ADDRESS': '하월곡동 96-113', 'TELEPHONE': '02-090-8083', 'APPROVAL_DATE': '33289', 'COMPLE_DATE': '1990.12.31'}, {'IDX': 135.0, 'FAC_NM': '월곡동아파트형공장', 'ADDRESS': '하월곡동 96-113', 'TELEPHONE': '02-912-1034', 'APPROVAL_DATE': '33289', 'COMPLE_DATE': '1990.12.31'}, {'IDX': 140.0, 'FAC_NM': '문정아르페온 1차 지식산업센터', 'ADDRESS': '서울특별시 송파구 문정동 640\n(문정도시개발구역 5-1BL)', 'TELEPHONE': '031-706-8577', 'APPROVAL_DATE': '2016.06.27', 'COMPLE_DATE': '2016.06.27'}, {'IDX': 139.0, 'FAC_NM': '문정 대명벨리온', 'ADDRESS': '서울특별시 송파구 문정동 641-4\n(문정도시개발구역 4-3BL)', 'TELEPHONE': '02-2222-7357', 'APPROVAL_DATE': '2016.03.23', 'COMPLE_DATE': '2016.03.23'}, {'IDX': 141.0, 'FAC_NM': '문정 현대지식산업센터Ｉ-1', 'ADDRESS': '서울특별시 송파구 문정동 644 (문정 특별계획구역 6-1BL)', 'TELEPHONE': '02-3451-0100', 'APPROVAL_DATE': '2016.02.29', 'COMPLE_DATE': '2016.02.29'}, {'IDX':

,IDX,FAC_NM,ADDRESS,TELEPHONE,APPROVAL_DATE,COMPLE_DATE
0,208.0,신내테크노타운,봉화산로 123(상봉동),02)3423-0057,96.5.28,99.10.18
1,136.0,월곡동아파트형공장,하월곡동 96-113,02-090-8083,33289,1990.12.31
2,135.0,월곡동아파트형공장,하월곡동 96-113,02-912-1034,33289,1990.12.31
3,140.0,문정아르페온 1차 지식산업센터,서울특별시 송파구 문정동 640\n(문정도시개발구역 5-1BL),031-706-8577,2016.06.27,2016.06.27
4,139.0,문정 대명벨리온,서울특별시 송파구 문정동 641-4\n(문정도시개발구역 4-3BL),02-2222-7357,2016.03.23,2016.03.23
...,...,...,...,...,...,...
208,169.0,서울제일인쇄협동조합,성수2가제3동 277-7번지,(02) 498-8727,1992.02.06,1993.05.08
209,168.0,삼풍,성수2가제3동 284-50번지,(02) 466-4515,1991.01.31,1992.08.03
210,122.0,대동아파트형공장,독산동 999-3,804-3362,1990.12,1992.02
211,3.0,비젼타워관리조합,양천로57길26,2659-0360,1990.10.31,1992.11.30


## 데이터 전처리

In [134]:
# 필요한 데이터만 추출하고, 열이름 모두 소문자로 통일 및 변경
# 1) 소문자로 변경
center_df = center_df[['IDX', 'FAC_NM', "ADDRESS"]]
center_df.columns = center_df.columns.str.lower()

center_df

# 2) 열이름 변경
center_df = center_df.rename(columns={
    'fac_nm': 'nm',
    'address': 'addr',
})

In [135]:
# 열 타입 변경
center_df['idx'] = center_df['idx'].astype("int")
center_df

,idx,nm,addr
0,208,신내테크노타운,봉화산로 123(상봉동)
1,136,월곡동아파트형공장,하월곡동 96-113
2,135,월곡동아파트형공장,하월곡동 96-113
3,140,문정아르페온 1차 지식산업센터,서울특별시 송파구 문정동 640\n(문정도시개발구역 5-1BL)
4,139,문정 대명벨리온,서울특별시 송파구 문정동 641-4\n(문정도시개발구역 4-3BL)
...,...,...,...
208,169,서울제일인쇄협동조합,성수2가제3동 277-7번지
209,168,삼풍,성수2가제3동 284-50번지
210,122,대동아파트형공장,독산동 999-3
211,3,비젼타워관리조합,양천로57길26


In [136]:
# 공장 이름과 주소가 모두 같은 행이 있는지 확인
print(center_df.duplicated(subset=['nm', 'addr']).sum()) # 1

dup_all_mask = center_df.duplicated(subset=['nm', 'addr'], keep=False)
center_df[dup_all_mask].sort_values(['nm', 'addr'])

1


,idx,nm,addr
1,136,월곡동아파트형공장,하월곡동 96-113
2,135,월곡동아파트형공장,하월곡동 96-113


In [137]:
# 공장 이름과 주소 중복 행 제거
center_df = center_df.drop_duplicates(subset=['nm', 'addr'], keep='first').reset_index(drop=True)
center_df

,idx,nm,addr
0,208,신내테크노타운,봉화산로 123(상봉동)
1,136,월곡동아파트형공장,하월곡동 96-113
2,140,문정아르페온 1차 지식산업센터,서울특별시 송파구 문정동 640\n(문정도시개발구역 5-1BL)
3,139,문정 대명벨리온,서울특별시 송파구 문정동 641-4\n(문정도시개발구역 4-3BL)
4,141,문정 현대지식산업센터Ｉ-1,서울특별시 송파구 문정동 644 (문정 특별계획구역 6-1BL)
...,...,...,...
207,169,서울제일인쇄협동조합,성수2가제3동 277-7번지
208,168,삼풍,성수2가제3동 284-50번지
209,122,대동아파트형공장,독산동 999-3
210,3,비젼타워관리조합,양천로57길26


In [138]:
# 주소 이상치 제거
center_df["addr"] = (
    center_df["addr"]
    .str.replace(r"(,|\n|외).*$", "", regex=True)  # 첫 , 또는 \n 또는 외 이후 전부 삭제
    .str.strip()
)

center_df

,idx,nm,addr
0,208,신내테크노타운,봉화산로 123(상봉동)
1,136,월곡동아파트형공장,하월곡동 96-113
2,140,문정아르페온 1차 지식산업센터,서울특별시 송파구 문정동 640
3,139,문정 대명벨리온,서울특별시 송파구 문정동 641-4
4,141,문정 현대지식산업센터Ｉ-1,서울특별시 송파구 문정동 644 (문정 특별계획구역 6-1BL)
...,...,...,...
207,169,서울제일인쇄협동조합,성수2가제3동 277-7번지
208,168,삼풍,성수2가제3동 284-50번지
209,122,대동아파트형공장,독산동 999-3
210,3,비젼타워관리조합,양천로57길26


In [139]:
print(center_df['addr'].str.contains(',').sum()) # 외 = 0, \n = 0, , = 1
print(center_df[center_df['addr'].str.contains(',', na=False)]) # idx = 202

cond = center_df['idx'] == 202
center_df.loc[cond, 'addr'] = '성수동2가 269-25' # 첫 주소로 대체

center_df.loc[cond, :]

1
    idx              nm                  addr
19  202  에이스하이앤드\n 성수타워  성수동2가 269-25,269-51,


,idx,nm,addr
19,202,에이스하이앤드\n 성수타워,성수동2가 269-25


In [140]:
# type 변수 추가
center_df['type'] = '지식산업센터'
center_df

,idx,nm,addr,type
0,208,신내테크노타운,봉화산로 123(상봉동),지식산업센터
1,136,월곡동아파트형공장,하월곡동 96-113,지식산업센터
2,140,문정아르페온 1차 지식산업센터,서울특별시 송파구 문정동 640,지식산업센터
3,139,문정 대명벨리온,서울특별시 송파구 문정동 641-4,지식산업센터
4,141,문정 현대지식산업센터Ｉ-1,서울특별시 송파구 문정동 644 (문정 특별계획구역 6-1BL),지식산업센터
...,...,...,...,...
207,169,서울제일인쇄협동조합,성수2가제3동 277-7번지,지식산업센터
208,168,삼풍,성수2가제3동 284-50번지,지식산업센터
209,122,대동아파트형공장,독산동 999-3,지식산업센터
210,3,비젼타워관리조합,양천로57길26,지식산업센터


### 주소 → 위도, 경도

In [21]:
kakao_key  = os.getenv("KAKAO_REST_APIKEY")     # 인증키

In [52]:
# test
url = "	https://dapi.kakao.com/v2/local/search/address.json" #요청할 url 주소
headers = {"Authorization": f"KakaoAK {kakao_key}"} #REST API 키(유효한 키)
params = {"query": "성수동2가 269-25"}  # 입력할 주소
resp = requests.get(url,headers=headers, params=params).json() #카카오 API 요청
                      
print(resp) #API 결과 출력
resp['documents'][0]['road_address']

{'documents': [{'address': {'address_name': '서울 성동구 성수동2가 269-25', 'b_code': '1120011500', 'h_code': '1120067000', 'main_address_no': '269', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '성동구', 'region_3depth_h_name': '성수2가1동', 'region_3depth_name': '성수동2가', 'sub_address_no': '25', 'x': '127.056741837461', 'y': '37.5402196364231'}, 'address_name': '서울 성동구 성수동2가 269-25', 'address_type': 'REGION_ADDR', 'road_address': {'address_name': '서울 성동구 성수이로10길 14', 'building_name': '에이스 하이엔드 성수타워', 'main_building_no': '14', 'region_1depth_name': '서울', 'region_2depth_name': '성동구', 'region_3depth_name': '성수동2가', 'road_name': '성수이로10길', 'sub_building_no': '', 'underground_yn': 'N', 'x': '127.056789858953', 'y': '37.5401307747523', 'zone_no': '04784'}, 'x': '127.056741837461', 'y': '37.5402196364231'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}


{'address_name': '서울 성동구 성수이로10길 14',
 'building_name': '에이스 하이엔드 성수타워',
 'main_building_no': '14',
 'region_1depth_name': '서울',
 'region_2depth_name': '성동구',
 'region_3depth_name': '성수동2가',
 'road_name': '성수이로10길',
 'sub_building_no': '',
 'underground_yn': 'N',
 'x': '127.056789858953',
 'y': '37.5401307747523',
 'zone_no': '04784'}

In [141]:
center_df = add_kakao_geocode(center_df, kakao_key)

In [142]:
print(center_df['rd_addr'].str.contains('서울').sum())

not_seoul = center_df[~center_df['rd_addr'].str.contains('서울')]
not_seoul

210


,idx,nm,addr,type,rd_addr,lat,lng
152,77,우림라이온스밸리1차,가산동 371-62,지식산업센터,NaN,NaN,NaN
194,48,천강,신도림동 402-3호,지식산업센터,NaN,NaN,NaN


In [143]:
'''
77	우림라이온스밸리1차	가산동 371-62
48	천강	신도림동 402-3호
두 주소는 이상치로 보고 제거.
'''
center_df.dropna(inplace=True)
len(center_df)

210

In [145]:
# lat, lng 결측치 확인. coerce: 변환 불가 시 NaN으로 바뀜
center_df["lat"] = pd.to_numeric(center_df["lat"], errors="coerce")
center_df["lng"] = pd.to_numeric(center_df["lng"], errors="coerce")
center_df[["lat", "lng"]].isna().sum()

lat    0
lng    0
dtype: int64

In [146]:
# 저장할 데이터 재추출
save_center_df = center_df[['nm', 'type', 'rd_addr', 'lat', 'lng']]

# 열이름 변경
save_center_df = save_center_df.rename(columns={'rd_addr': 'addr'})

# index 재설정
save_center_df.reset_index(drop=True, inplace=True) 

save_center_df.head()

,nm,type,addr,lat,lng
0,신내테크노타운,지식산업센터,서울 중랑구 봉화산로 123,37.604027,127.087796
1,월곡동아파트형공장,지식산업센터,서울 성북구 오패산로 21,37.604119,127.037068
2,문정아르페온 1차 지식산업센터,지식산업센터,서울 송파구 정의로 70,37.486048,127.115468
3,문정 대명벨리온,지식산업센터,서울 송파구 법원로 127,37.486316,127.118783
4,문정 현대지식산업센터Ｉ-1,지식산업센터,서울 송파구 법원로11길 11,37.484630,127.118322


# 서울시 휴게음식점 인허가

## 데이터 불러오기

In [5]:
apikey  = os.getenv("MJ_APIKEY")     # 인증키
start_index = "1"    # 요청시작위치
end_index   = "1000"  # 요청종료위치

url = f"http://openapi.seoul.go.kr:8088/{apikey}/json/LOCALDATA_072405/{start_index}/{end_index}/"

response = requests.get(url).json()
print(response)

{'LOCALDATA_072405': {'list_total_count': 143069, 'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다'}, 'row': [{'OPNSFTEAMCODE': '3040000', 'MGTNO': '3040000-104-2016-00210', 'APVPERMYMD': '2016-09-21', 'APVCANCELYMD': '', 'TRDSTATEGBN': '03', 'TRDSTATENM': '폐업', 'DTLSTATEGBN': '02', 'DTLSTATENM': '폐업', 'DCBYMD': '2023-04-26', 'CLGSTDT': '', 'CLGENDDT': '', 'ROPNYMD': '', 'SITETEL': '', 'SITEAREA': '31.17', 'SITEPOSTNO': '143-822', 'SITEWHLADDR': '서울특별시 광진구 구의동 212-3 1층 ', 'RDNWHLADDR': '서울특별시 광진구 아차산로 471, 1층 (구의동, CS프라자)', 'RDNPOSTNO': '05035', 'BPLCNM': '자연드림 구의점', 'LASTMODTS': '2023-04-26 16:50:33', 'UPDATEGBN': 'U', 'UPDATEDT': '2023-04-28 02:40:00.0', 'UPTAENM': '기타 휴게음식점', 'X': '208232.99266682     ', 'Y': '448688.329238784    ', 'SNTUPTAENM': '기타 휴게음식점', 'MANEIPCNT': '0', 'WMEIPCNT': '0', 'TRDPJUBNSENM': '', 'LVSENM': '', 'WTRSPLYFACILSENM': '', 'TOTEPNUM': '0', 'HOFFEPCNT': '0', 'FCTYOWKEPCNT': '0', 'FCTYSILJOBEPCNT': '0', 'FCTYPDTJOBEPCNT': '0', 'BDNGOWNSENM': '', 'ISREAM

In [6]:
# 'list_total_count': 142943 ⇒ 1000개씩 나눠서 가져와야 함
apikey = os.getenv("MJ_APIKEY")
service = "LOCALDATA_072405"
base = "http://openapi.seoul.go.kr:8088"

# 1) 전체 건수 확인
first = requests.get(f"{base}/{apikey}/json/{service}/1/1/").json()
total = first[service]["list_total_count"]

# 2) 분할 수집
batch = 1000   # 서비스 허용 범위 내에서 조절
all_rows = []

for start in range(1, total + 1, batch):
    end = min(start + batch - 1, total)
    url = f"{base}/{apikey}/json/{service}/{start}/{end}/"
    response = requests.get(url).json()
    rows = response.get(service, {}).get("row", [])
    all_rows.extend(rows)

print("total:", total)
print("fetched:", len(all_rows))

total: 143069
fetched: 143069


In [7]:
cafe_df = pd.DataFrame(all_rows)
cafe_df.head()

,OPNSFTEAMCODE,MGTNO,APVPERMYMD,APVCANCELYMD,TRDSTATEGBN,TRDSTATENM,DTLSTATEGBN,DTLSTATENM,DCBYMD,CLGSTDT,...,FCTYSILJOBEPCNT,FCTYPDTJOBEPCNT,BDNGOWNSENM,ISREAM,MONAM,MULTUSNUPSOYN,FACILTOTSCP,JTUPSOASGNNO,JTUPSOMAINEDF,HOMEPAGE
0,3040000,3040000-104-2016-00210,2016-09-21,,03,폐업,02,폐업,2023-04-26,,...,0,0,,0,0,N,31.17,,,
1,3040000,3040000-104-2016-00212,2016-09-27,,03,폐업,02,폐업,2018-03-23,,...,,,,,,N,64,,,
2,3040000,3040000-104-2016-00213,2016-09-28,,03,폐업,02,폐업,2023-11-02,,...,0,0,,0,0,N,10,,,
3,3040000,3040000-104-2016-00214,2016-09-29,,03,폐업,02,폐업,2021-11-08,,...,0,0,,0,0,N,6.6,,,
4,3040000,3040000-104-2018-00033,2018-02-01,,03,폐업,02,폐업,2023-08-18,,...,0,0,,0,0,N,6.6,,,


## 데이터 전처리

In [8]:
# 필요한 데이터만 추출하고, 열이름 모두 소문자로 통일
cafe_df = cafe_df[['BPLCNM', "UPTAENM", 'DTLSTATEGBN', "APVPERMYMD", "SITEWHLADDR", "RDNWHLADDR", "SITEAREA", "LASTMODTS"]]
cafe_df.columns = cafe_df.columns.str.lower()

cafe_df

,bplcnm,uptaenm,dtlstategbn,apvpermymd,sitewhladdr,rdnwhladdr,sitearea,lastmodts
0,자연드림 구의점,기타 휴게음식점,02,2016-09-21,서울특별시 광진구 구의동 212-3 1층,"서울특별시 광진구 아차산로 471, 1층 (구의동, CS프라자)",31.17,2023-04-26 16:50:33
1,커피뎁(coffee dep),커피숍,02,2016-09-27,서울특별시 광진구 중곡동 629-7번지 1층,"서울특별시 광진구 면목로 59, 1층 (중곡동)",,2018-03-23 14:00:12
2,엔탑피씨방,일반조리판매,02,2016-09-28,서울특별시 광진구 능동 279-5 삼일빌딩 지하1층,"서울특별시 광진구 능동로 290, 지1층 (능동, 삼일빌딩)",10.00,2023-11-02 10:31:12
3,지에스25구의래미안,기타 휴게음식점,02,2016-09-29,서울특별시 광진구 구의동 65-10 1층,"서울특별시 광진구 광나루로37길 21, 1층 (구의동)",6.60,2021-11-08 08:12:13
4,씨유중곡오거리점,편의점,02,2018-02-01,서울특별시 광진구 중곡동 150-160 종호빌딩1층,"서울특별시 광진구 긴고랑로25길 29, 종호빌딩 1층 (중곡동)",6.60,2023-08-18 13:20:53
...,...,...,...,...,...,...,...,...
143064,뎁스Depth,커피숍,01,2020-09-15,서울특별시 강서구 화곡동 55-59 1층,"서울특별시 강서구 초록마을로 56, 1층 (화곡동)",94.84,2026-02-27 09:00:48
143065,부산빨간어묵포차 수색점,일반조리판매,02,2019-07-12,서울특별시 은평구 수색동 75 디엠씨 자이1단지(DMC 자이1) 2층,"서울특별시 은평구 수색로 217, 2층 (수색동, 디엠씨 자이1단지(DMC 자이1))",15.84,2026-02-27 14:58:11
143066,계순네 닭강정,기타 휴게음식점,01,2016-12-26,,"서울특별시 마포구 백범로 68 (신수동, 1층일부)",9.90,2026-02-27 14:55:08
143067,장손족발(영등포점),일반조리판매,02,2015-04-07,서울특별시 영등포구 문래동3가 55-3 홈플러스영등포점 지하 1층,"서울특별시 영등포구 당산로 42 (문래동3가, 홈플러스영등포점 지하 1층)",8.20,2026-02-27 10:13:04


In [8]:
cafe_df['uptaenm'].unique()

<StringArray>
['기타 휴게음식점',      '커피숍',   '일반조리판매',      '편의점',    '패스트푸드',      '과자점',
     '푸드트럭',       '다방',     '관광호텔',      '유원지',    '철도역구내',      '백화점',
      '떡카페',    '아이스크림',     '키즈카페',     '전통찻집',       '극장',       '공항',
     '단란주점',    '호프/통닭',     '고속도로',       '기타',  '김밥(도시락)',      '룸살롱',
       '한식']
Length: 25, dtype: str

In [9]:
# '영업' 중이면서 '커피숍'인 데이터만 필터링
cafe_type = ['커피숍']
cond = (cafe_df['uptaenm'].isin(cafe_type)) & (cafe_df['dtlstategbn'] == '01')  # 01=영업

cafe_df = cafe_df.loc[cond, :]
cafe_df

,bplcnm,uptaenm,dtlstategbn,apvpermymd,sitewhladdr,rdnwhladdr,sitearea,lastmodts
220,이디야답십리파크자이점,커피숍,01,2019-03-11,서울특별시 동대문구 답십리동 12-18 영진빌딩,"서울특별시 동대문구 한천로 55, 영진빌딩 1층 (답십리동)",85.00,2022-07-07 14:22:12
223,다락방 사주카페,커피숍,01,2014-04-09,서울특별시 동대문구 휘경동 320-5번지,서울특별시 동대문구 회기로 185 (휘경동),49.59,2019-04-03 15:45:38
225,빽다방 전농동전동초교점,커피숍,01,2023-08-23,서울특별시 동대문구 전농동 38-112,"서울특별시 동대문구 사가정로 145, 1층 (전농동)",57.00,2024-02-02 13:26:49
241,커피상회,커피숍,01,2023-08-18,서울특별시 동대문구 청량리동 225-4,"서울특별시 동대문구 약령시로 133, 1층 (청량리동)",9.90,2023-08-18 13:24:50
245,구하니 커피,커피숍,01,2023-08-04,서울특별시 동대문구 전농동 295-431,"서울특별시 동대문구 전농로15길 21, 남동 2호 (전농동)",10.36,2024-09-11 10:45:00
...,...,...,...,...,...,...,...,...
143030,CornerStone Espresso Bar,커피숍,01,2026-02-27,서울특별시 강서구 마곡동 771-3 보타닉파크타워3,"서울특별시 강서구 마곡중앙6로 11, 보타닉파크타워3 1층 107호 (마곡동)",39.90,2026-02-27 10:35:12
143041,매머드커피 익스프레스 선릉501스퀘어점,커피숍,01,2026-02-27,서울특별시 강남구 삼성동 143-32,"서울특별시 강남구 삼성로91길 32, 1층 (삼성동)",72.54,2026-02-27 12:30:31
143060,카페 선공간,커피숍,01,2023-05-17,서울특별시 동작구 신대방동 686,"서울특별시 동작구 대림로 61, 1층 (신대방동)",26.44,2026-02-27 11:52:07
143061,포트캔커피 상일동역점,커피숍,01,2022-07-21,서울특별시 강동구 고덕동 693 고덕그라시움(제1상가),"서울특별시 강동구 고덕로 353, 고덕그라시움(제1상가) 1층 에이137호 (고덕동)",27.55,2026-02-27 14:39:07


In [10]:
# 공백/빈문자열을 전부 NA로 통일
cafe_df = cafe_df.replace(r'^\s*$', pd.NA, regex=True)

In [11]:
# 결측치 확인
cafe_df.isna().sum()

bplcnm           0
uptaenm          0
dtlstategbn      0
apvpermymd       0
sitewhladdr     10
rdnwhladdr      72
sitearea       286
lastmodts        0
dtype: int64

In [12]:
# sitewhladdr(지번주소)가 NA인 곳만 rdnwhladdr(도로명주소)값으로 채워 결측치 없애기
cafe_df["sitewhladdr"] = cafe_df["sitewhladdr"].fillna(cafe_df["rdnwhladdr"])
cafe_df.isna().sum()

bplcnm           0
uptaenm          0
dtlstategbn      0
apvpermymd       0
sitewhladdr      0
rdnwhladdr      72
sitearea       286
lastmodts        0
dtype: int64

In [13]:
# 층수 추출하기
pat_block = re.compile(r'(지하|지상)?\s*(\d+(?:\s*,\s*\d+)*)\s*층')
pat_num = re.compile(r'\d+')
pat_ho = re.compile(r'(\d{3,4})\s*호')  # 101호, 1101호

def get_floor(addr):
    if pd.isna(addr):
        return pd.NA

    text = str(addr)
    vals = []

    # 1) 일반 층수 패턴
    for prefix, nums in pat_block.findall(text):
        for n in pat_num.findall(nums):
            v = int(n)
            vals.append(-v if prefix == '지하' else v)

    # 2) 층 정보가 없으면 호수에서 추정
    if not vals:
        for h in pat_ho.findall(text):
            if len(h) == 3:      # 101호 -> 1층
                vals.append(int(h[0]))
            elif len(h) == 4:    # 1101호 -> 11층
                vals.append(int(h[:2]))

    if not vals:
        return pd.NA
    if 1 in vals:
        return 1

    positives = any(v > 0 for v in vals)
    negatives = any(v < 0 for v in vals)
    if negatives and not positives:
        return -1

    return min(vals)

cafe_df["floor"] = cafe_df["sitewhladdr"].apply(get_floor).astype("Int64")
cafe_df["floor"] = cafe_df["floor"].fillna(
    cafe_df["rdnwhladdr"].apply(get_floor).astype("Int64")
)
cafe_df

,bplcnm,uptaenm,dtlstategbn,apvpermymd,sitewhladdr,rdnwhladdr,sitearea,lastmodts,floor
220,이디야답십리파크자이점,커피숍,01,2019-03-11,서울특별시 동대문구 답십리동 12-18 영진빌딩,"서울특별시 동대문구 한천로 55, 영진빌딩 1층 (답십리동)",85.00,2022-07-07 14:22:12,1
223,다락방 사주카페,커피숍,01,2014-04-09,서울특별시 동대문구 휘경동 320-5번지,서울특별시 동대문구 회기로 185 (휘경동),49.59,2019-04-03 15:45:38,<NA>
225,빽다방 전농동전동초교점,커피숍,01,2023-08-23,서울특별시 동대문구 전농동 38-112,"서울특별시 동대문구 사가정로 145, 1층 (전농동)",57.00,2024-02-02 13:26:49,1
241,커피상회,커피숍,01,2023-08-18,서울특별시 동대문구 청량리동 225-4,"서울특별시 동대문구 약령시로 133, 1층 (청량리동)",9.90,2023-08-18 13:24:50,1
245,구하니 커피,커피숍,01,2023-08-04,서울특별시 동대문구 전농동 295-431,"서울특별시 동대문구 전농로15길 21, 남동 2호 (전농동)",10.36,2024-09-11 10:45:00,<NA>
...,...,...,...,...,...,...,...,...,...
143030,CornerStone Espresso Bar,커피숍,01,2026-02-27,서울특별시 강서구 마곡동 771-3 보타닉파크타워3,"서울특별시 강서구 마곡중앙6로 11, 보타닉파크타워3 1층 107호 (마곡동)",39.90,2026-02-27 10:35:12,1
143041,매머드커피 익스프레스 선릉501스퀘어점,커피숍,01,2026-02-27,서울특별시 강남구 삼성동 143-32,"서울특별시 강남구 삼성로91길 32, 1층 (삼성동)",72.54,2026-02-27 12:30:31,1
143060,카페 선공간,커피숍,01,2023-05-17,서울특별시 동작구 신대방동 686,"서울특별시 동작구 대림로 61, 1층 (신대방동)",26.44,2026-02-27 11:52:07,1
143061,포트캔커피 상일동역점,커피숍,01,2022-07-21,서울특별시 강동구 고덕동 693 고덕그라시움(제1상가),"서울특별시 강동구 고덕로 353, 고덕그라시움(제1상가) 1층 에이137호 (고덕동)",27.55,2026-02-27 14:39:07,1


In [14]:
# 1) 열이름 변경
cafe_df = cafe_df.rename(columns={
    'bplcnm': 'nm',
    'uptaenm': 'type',
    'dtlstategbn': 'stable',
    'apvpermymd': 'apv_date',
    'sitewhladdr': 'addr',
    'rdnwhladdr': 'rd_addr',
    'lastmodts': 'last_updated',
})

# 2) 열 순서 재배치
cafe_df = cafe_df[['nm', 'type', 'apv_date', 'addr', 'rd_addr', 'floor', 'sitearea', 'last_updated']]

In [15]:
# 결측치 확인
cafe_df.isna().sum()

nm                0
type              0
apv_date          0
addr              0
rd_addr          72
floor           626
sitearea        286
last_updated      0
dtype: int64

In [16]:
# 주소 이상치 제거 for 위도, 경도 탐색
cafe_df["addr"] = (
    cafe_df["addr"]
    .str.replace(r"\s*(?:외\s*)?\d+\s*필지'?", "", regex=True)  # 외 2필지 / 2필지 / 2 필지 / 외2필지'
    .str.strip()
)
cafe_df

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated
220,이디야답십리파크자이점,커피숍,2019-03-11,서울특별시 동대문구 답십리동 12-18 영진빌딩,"서울특별시 동대문구 한천로 55, 영진빌딩 1층 (답십리동)",1,85.00,2022-07-07 14:22:12
223,다락방 사주카페,커피숍,2014-04-09,서울특별시 동대문구 휘경동 320-5번지,서울특별시 동대문구 회기로 185 (휘경동),<NA>,49.59,2019-04-03 15:45:38
225,빽다방 전농동전동초교점,커피숍,2023-08-23,서울특별시 동대문구 전농동 38-112,"서울특별시 동대문구 사가정로 145, 1층 (전농동)",1,57.00,2024-02-02 13:26:49
241,커피상회,커피숍,2023-08-18,서울특별시 동대문구 청량리동 225-4,"서울특별시 동대문구 약령시로 133, 1층 (청량리동)",1,9.90,2023-08-18 13:24:50
245,구하니 커피,커피숍,2023-08-04,서울특별시 동대문구 전농동 295-431,"서울특별시 동대문구 전농로15길 21, 남동 2호 (전농동)",<NA>,10.36,2024-09-11 10:45:00
...,...,...,...,...,...,...,...,...
143030,CornerStone Espresso Bar,커피숍,2026-02-27,서울특별시 강서구 마곡동 771-3 보타닉파크타워3,"서울특별시 강서구 마곡중앙6로 11, 보타닉파크타워3 1층 107호 (마곡동)",1,39.90,2026-02-27 10:35:12
143041,매머드커피 익스프레스 선릉501스퀘어점,커피숍,2026-02-27,서울특별시 강남구 삼성동 143-32,"서울특별시 강남구 삼성로91길 32, 1층 (삼성동)",1,72.54,2026-02-27 12:30:31
143060,카페 선공간,커피숍,2023-05-17,서울특별시 동작구 신대방동 686,"서울특별시 동작구 대림로 61, 1층 (신대방동)",1,26.44,2026-02-27 11:52:07
143061,포트캔커피 상일동역점,커피숍,2022-07-21,서울특별시 강동구 고덕동 693 고덕그라시움(제1상가),"서울특별시 강동구 고덕로 353, 고덕그라시움(제1상가) 1층 에이137호 (고덕동)",1,27.55,2026-02-27 14:39:07


In [17]:
# 중복 확인
print(cafe_df.duplicated(subset=['nm', 'addr']).sum()) 

dup_all_mask = cafe_df.duplicated(subset=['nm', 'addr'], keep=False)
cafe_df[dup_all_mask].sort_values(['nm', 'addr'])

15


,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated
86218,공차 영등포역점,커피숍,2022-07-12,서울특별시 영등포구 영등포동3가 9-51,"서울특별시 영등포구 영중로 10, 1층 (영등포동3가)",1,51.50,2025-02-25 13:33:03
86219,공차 영등포역점,커피숍,2022-07-12,서울특별시 영등포구 영등포동3가 9-51,"서울특별시 영등포구 영중로 10, 2층 (영등포동3가)",2,51.50,2025-02-25 13:26:48
122639,누구나카페,커피숍,2025-07-08,서울특별시 마포구 성산동 370 마포구청,"서울특별시 마포구 월드컵로 212, 마포구청 지하1층 (성산동)",-1,31.50,2025-07-08 17:21:46
141619,누구나카페,커피숍,2026-01-08,서울특별시 마포구 성산동 370 마포구청,"서울특별시 마포구 월드컵로 212, 마포구보건소 1층 (성산동)",1,15.50,2026-01-08 14:04:52
78234,라이프플러스 카페(LIFEPLUS CAFE),커피숍,2023-01-03,서울특별시 영등포구 여의도동 60 63한화생명빌딩,"서울특별시 영등포구 63로 50, 63한화생명빌딩 7층 (여의도동)",7,23.55,2025-08-18 14:34:18
78235,라이프플러스 카페(LIFEPLUS CAFE),커피숍,2023-01-03,서울특별시 영등포구 여의도동 60 63한화생명빌딩,"서울특별시 영등포구 63로 50, 63한화생명빌딩 45층 (여의도동)",45,28.34,2025-08-18 14:35:12
86441,마이유(My U),커피숍,2022-12-09,서울특별시 영등포구 신길동 51-1,"서울특별시 영등포구 영등포로 349, 1층 (신길동)",1,42.43,2022-12-09 15:52:39
86442,마이유(My U),커피숍,2022-12-09,서울특별시 영등포구 신길동 51-1,"서울특별시 영등포구 영등포로 349, 2층 (신길동)",2,42.43,2022-12-09 15:56:25
121382,산체스커피,커피숍,2024-08-09,서울특별시 송파구 오금동 61-10 동성빌딩,"서울특별시 송파구 성내천로8길 3, 동성빌딩 1층 1,2호 (오금동)",1,62.00,2024-10-11 17:35:40
122491,산체스커피,커피숍,2023-02-28,서울특별시 송파구 오금동 61-10 동성빌딩,"서울특별시 송파구 성내천로8길 3, 동성빌딩 1층 3호 (오금동)",1,25.00,2025-10-02 09:16:29


In [18]:
# 최신 업데이트 정보 기준으로 중복제거
# 1) datetime 변환
cafe_df["last_updated"] = pd.to_datetime(cafe_df["last_updated"], errors="coerce")

# 2) 같은 (cafe_nm, address) 그룹에서 최신 last_updated만 남기기
cafe_df = (
    cafe_df.sort_values("last_updated")  # 오래된 -> 최신
           .drop_duplicates(subset=["nm", "addr"], keep="last")
           .reset_index(drop=True)
)

# 확인
print(cafe_df.duplicated(subset=["nm", "addr"]).sum())

0


In [19]:
# 중복 제거 잘 되었는지 케이스 하나 확인
cafe_df.loc[(cafe_df['nm'] == '누구나카페'), :]

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated
14076,누구나카페,커피숍,2026-01-08,서울특별시 마포구 성산동 370 마포구청,"서울특별시 마포구 월드컵로 212, 마포구보건소 1층 (성산동)",1,15.50,2026-01-08 14:04:52


### 주소 → 위도, 경도

In [20]:
kakao_key  = os.getenv("KAKAO_REST_APIKEY")     # 인증키

In [21]:
cafe_df = add_kakao_geocode(cafe_df, kakao_key, address_col="addr", out_col="kakao_addr")
cafe_df

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated,kakao_addr,lat,lng
0,베레베레,커피숍,2002-09-19,서울특별시 광진구 화양동 11-2번지,서울특별시 광진구 능동로13길 19 (화양동),<NA>,NaN,2002-09-19 00:00:00,서울 광진구 능동로13길 19,37.542885,127.070832
1,클럽알앤비,커피숍,2002-11-05,서울특별시 중구 장충동2가 189-2번지 (동대입구역322-13),NaN,<NA>,17.00,2002-11-05 00:00:00,NaN,NaN,NaN
2,덕수궁전통차전문,커피숍,2003-01-16,서울특별시 중구 서소문동 54-1번지 (2층),"서울특별시 중구 서소문로 109 (서소문동,(2층))",2,46.28,2003-01-22 00:00:00,서울 중구 서소문로 109,37.563062,126.973280
3,엘빠소,커피숍,2003-05-31,서울특별시 종로구 명륜2가 143-9번지,서울특별시 종로구 성균관로 18 (명륜2가),<NA>,NaN,2003-05-31 00:00:00,서울 종로구 성균관로 18,37.584406,126.997641
4,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울특별시 종로구 세종로 82-14번지 지상1층,"서울특별시 종로구 세종대로 188 (세종로,지상1층)",1,NaN,2003-11-21 00:00:00,서울 종로구 세종대로 188,37.573190,126.977831
...,...,...,...,...,...,...,...,...,...,...,...
14567,카페 라희,커피숍,2026-02-27,서울특별시 강동구 길동 474 GS강동자이아파트,"서울특별시 강동구 천호대로199길 10, 1층 103호 (길동, GS강동자이아파트)",1,25.20,2026-02-27 15:32:32,서울 강동구 천호대로 1239,37.537861,127.148277
14568,달콤한 오늘,커피숍,2026-02-27,서울특별시 강서구 마곡동 739-3 에스비타운,"서울특별시 강서구 마곡중앙5로 81, 에스비타운 1층 115호 (마곡동)",1,62.33,2026-02-27 16:28:56,서울 강서구 마곡서1로 111-12,37.566787,126.817788
14569,쿠즈코커피 마포성산점,커피숍,2026-02-23,서울특별시 마포구 성산동 231-4,"서울특별시 마포구 월드컵북로 73, 1층 (성산동)",1,35.43,2026-02-27 16:46:41,서울 마포구 월드컵북로 73,37.560317,126.915972
14570,푸디스트(주)서울시중구청카페,커피숍,2026-02-27,서울특별시 중구 예관동 120-1 중구청,"서울특별시 중구 창경궁로 17, 중구청 1층 (예관동)",1,11.94,2026-02-27 17:15:09,서울 중구 창경궁로 17,37.564120,126.998010


In [22]:
# 결측치 확인. coerce: 변환 불가 시 NaN으로 바뀜
cafe_df["lat"] = pd.to_numeric(cafe_df["lat"], errors="coerce")
cafe_df["lng"] = pd.to_numeric(cafe_df["lng"], errors="coerce")
cafe_df[["kakao_addr", "lat", "lng"]].isna().sum()

kakao_addr    243
lat           282
lng           282
dtype: int64

In [23]:
# rd_addr 주소 이상치 제거 for 안정적인 위도, 경도 탐색
cafe_df["rd_addr"] = (
    cafe_df["rd_addr"]
    .str.replace(r"[,()].*$", "", regex=True) # rd_addr에서 ',' 또는 '('가 처음 나오는 지점부터 끝까지 삭제
    .str.strip()
)
cafe_df


,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated,kakao_addr,lat,lng
0,베레베레,커피숍,2002-09-19,서울특별시 광진구 화양동 11-2번지,서울특별시 광진구 능동로13길 19,<NA>,NaN,2002-09-19 00:00:00,서울 광진구 능동로13길 19,37.542885,127.070832
1,클럽알앤비,커피숍,2002-11-05,서울특별시 중구 장충동2가 189-2번지 (동대입구역322-13),NaN,<NA>,17.00,2002-11-05 00:00:00,NaN,NaN,NaN
2,덕수궁전통차전문,커피숍,2003-01-16,서울특별시 중구 서소문동 54-1번지 (2층),서울특별시 중구 서소문로 109,2,46.28,2003-01-22 00:00:00,서울 중구 서소문로 109,37.563062,126.973280
3,엘빠소,커피숍,2003-05-31,서울특별시 종로구 명륜2가 143-9번지,서울특별시 종로구 성균관로 18,<NA>,NaN,2003-05-31 00:00:00,서울 종로구 성균관로 18,37.584406,126.997641
4,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울특별시 종로구 세종로 82-14번지 지상1층,서울특별시 종로구 세종대로 188,1,NaN,2003-11-21 00:00:00,서울 종로구 세종대로 188,37.573190,126.977831
...,...,...,...,...,...,...,...,...,...,...,...
14567,카페 라희,커피숍,2026-02-27,서울특별시 강동구 길동 474 GS강동자이아파트,서울특별시 강동구 천호대로199길 10,1,25.20,2026-02-27 15:32:32,서울 강동구 천호대로 1239,37.537861,127.148277
14568,달콤한 오늘,커피숍,2026-02-27,서울특별시 강서구 마곡동 739-3 에스비타운,서울특별시 강서구 마곡중앙5로 81,1,62.33,2026-02-27 16:28:56,서울 강서구 마곡서1로 111-12,37.566787,126.817788
14569,쿠즈코커피 마포성산점,커피숍,2026-02-23,서울특별시 마포구 성산동 231-4,서울특별시 마포구 월드컵북로 73,1,35.43,2026-02-27 16:46:41,서울 마포구 월드컵북로 73,37.560317,126.915972
14570,푸디스트(주)서울시중구청카페,커피숍,2026-02-27,서울특별시 중구 예관동 120-1 중구청,서울특별시 중구 창경궁로 17,1,11.94,2026-02-27 17:15:09,서울 중구 창경궁로 17,37.564120,126.998010


In [25]:
# 결측치 값에 한해서 lat, lng 다시 채우기 
mask = cafe_df["lat"].isna() & cafe_df["rd_addr"].notna()

cafe_df.loc[mask, ["kakao_addr", "lat", "lng"]] = (
    add_kakao_geocode(
        cafe_df.loc[mask, ["rd_addr"]].copy(),   # (조회 기준은 rd_addr)
        kakao_key,
        address_col="rd_addr",
        out_col = "kakao_addr"
    )[["kakao_addr", "lat", "lng"]]
    .to_numpy()
)

In [26]:
# 결측치 재확인
cafe_df["lat"] = pd.to_numeric(cafe_df["lat"], errors="coerce")
cafe_df["lng"] = pd.to_numeric(cafe_df["lng"], errors="coerce")
cafe_df[["kakao_addr", "lat", "lng"]].isna().sum()

kakao_addr    39
lat           39
lng           39
dtype: int64

In [27]:
cafe_df[cafe_df["lat"].isna()]

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated,kakao_addr,lat,lng
1,클럽알앤비,커피숍,2002-11-05,서울특별시 중구 장충동2가 189-2번지 (동대입구역322-13),NaN,<NA>,17.00,2002-11-05 00:00:00,NaN,NaN,NaN
13,할리스커피,커피숍,2005-08-16,서울특별시 종로구 공평동 55번지 지상1층,서울특별시 종로구 우정국로 24,1,NaN,2005-08-16 00:00:00,NaN,NaN,NaN
27,원타임편의점,커피숍,2007-07-25,서울특별시 용산구 한강로2가 351번지 (지상1층),NaN,1,3.30,2007-07-25 11:07:04,NaN,NaN,NaN
28,이디야중대용산병원점,커피숍,2007-08-13,서울특별시 용산구 한강로3가 65-207번지 (지상1층),NaN,1,7.70,2007-08-13 16:57:29,NaN,NaN,NaN
66,이디야커피동국대점,커피숍,2009-08-27,서울특별시 중구 충무로5가 79-2번지 (지상1층),서울특별시 중구 서애로 7,1,24.00,2009-08-27 13:38:53,NaN,NaN,NaN
128,로뎀나무,커피숍,2010-08-13,서울특별시 노원구 월계동 287-11번지,서울특별시 노원구 월계로 337,<NA>,33.93,2011-03-17 10:39:19,NaN,NaN,NaN
139,함초美,커피숍,2011-03-18,"서울특별시 동작구 대방동 501번지 대림쇼핑상가동 1-21,24호",NaN,<NA>,16.50,2011-05-20 10:50:14,NaN,NaN,NaN
141,커피향,커피숍,2007-03-06,"서울특별시 금천구 시흥동 999번지 - 29,31,51 지상2층 (시흥대로 446)",서울특별시 금천구 시흥대로71길 7,2,26.45,2011-06-08 10:29:44,NaN,NaN,NaN
158,훼미리마트(이화사거리점),커피숍,2011-07-26,서울특별시 종로구 연건동 196-11번지 1층,NaN,1,NaN,2011-07-26 16:27:15,NaN,NaN,NaN
159,와플반트 제일병원점,커피숍,2011-08-01,서울특별시 중구 묵정동 1-2번지 외래센터건물 지상3층,서울특별시 중구 서애로1길 17,3,12.87,2011-08-01 14:15:07,NaN,NaN,NaN


In [28]:
cafe_df.dropna(subset=["kakao_addr"], inplace=True)
cafe_df

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated,kakao_addr,lat,lng
0,베레베레,커피숍,2002-09-19,서울특별시 광진구 화양동 11-2번지,서울특별시 광진구 능동로13길 19,<NA>,NaN,2002-09-19 00:00:00,서울 광진구 능동로13길 19,37.542885,127.070832
2,덕수궁전통차전문,커피숍,2003-01-16,서울특별시 중구 서소문동 54-1번지 (2층),서울특별시 중구 서소문로 109,2,46.28,2003-01-22 00:00:00,서울 중구 서소문로 109,37.563062,126.973280
3,엘빠소,커피숍,2003-05-31,서울특별시 종로구 명륜2가 143-9번지,서울특별시 종로구 성균관로 18,<NA>,NaN,2003-05-31 00:00:00,서울 종로구 성균관로 18,37.584406,126.997641
4,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울특별시 종로구 세종로 82-14번지 지상1층,서울특별시 종로구 세종대로 188,1,NaN,2003-11-21 00:00:00,서울 종로구 세종대로 188,37.573190,126.977831
5,본솔카페,커피숍,2004-01-17,서울특별시 용산구 청파동2가 64-13번지 청파3가 115-12 (지상1층),서울특별시 용산구 청파로47길 71,1,13.80,2004-01-17 00:00:00,서울 용산구 청파로47길 71,37.544840,126.966455
...,...,...,...,...,...,...,...,...,...,...,...
14567,카페 라희,커피숍,2026-02-27,서울특별시 강동구 길동 474 GS강동자이아파트,서울특별시 강동구 천호대로199길 10,1,25.20,2026-02-27 15:32:32,서울 강동구 천호대로 1239,37.537861,127.148277
14568,달콤한 오늘,커피숍,2026-02-27,서울특별시 강서구 마곡동 739-3 에스비타운,서울특별시 강서구 마곡중앙5로 81,1,62.33,2026-02-27 16:28:56,서울 강서구 마곡서1로 111-12,37.566787,126.817788
14569,쿠즈코커피 마포성산점,커피숍,2026-02-23,서울특별시 마포구 성산동 231-4,서울특별시 마포구 월드컵북로 73,1,35.43,2026-02-27 16:46:41,서울 마포구 월드컵북로 73,37.560317,126.915972
14570,푸디스트(주)서울시중구청카페,커피숍,2026-02-27,서울특별시 중구 예관동 120-1 중구청,서울특별시 중구 창경궁로 17,1,11.94,2026-02-27 17:15:09,서울 중구 창경궁로 17,37.564120,126.998010


### 유동인구 데이터 활용을 위한 지역 코드 추가

#### 상권 코드 추가
- 상권/상권 배후지/행정동 코드는 유동인구를 파악하기 위함
- 유동인구 데이터가 없다면, 카페 입지 분석이 어려움.
- 따라서 결측이가 있을 경우, **상권 → 상권 배후지 → 행정동** 순으로 유동 인구 정보를 활용해야 함

In [ ]:
# 결측치 확인
cafe_df[["lat", "lng"]].isna().sum()

lat    0
lng    0
dtype: int64

In [ ]:
# 1) cafe_df -> GeoDataFrame (실제 좌표계: WGS84)
cafe_gdf = gpd.GeoDataFrame(
    cafe_df.copy(),
    geometry=gpd.points_from_xy(cafe_df["lng"], cafe_df["lat"]),
    crs="EPSG:4326"
)

# 2) 상권 shp 로드 (좌표계: EPSG:5181)
trdar_gdf = gpd.read_file(
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-상권).zip"
)
# → 이 파일은 EPSG:5181이 이미 설정되어 있어야 함

# 3) 좌표계 통일 (cafe를 5181로 변환)
'''
폴리곤(상권)은 꼭짓점이 많아서 변환 시 오차가 누적될 수 있으므로 shp(5181) 기준으로 맞춤
'''
cafe_gdf = cafe_gdf.to_crs("EPSG:5181")  # shp 기준으로 맞춤

# 4) 공간조인
trdar_cols = ["TRDAR_CD", "geometry"]
joined = gpd.sjoin(
    cafe_gdf,
    trdar_gdf[trdar_cols],
    how="left",
    predicate="within"
)

# 5) 원본 인덱스 기준으로 TRDAR_CD 붙이기
cafe_df = cafe_df.copy()
cafe_df.loc[joined.index, "trdar_cd"] = joined["TRDAR_CD"].values

In [ ]:
cafe_df.head()

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated,kakao_addr,lat,lng,alley_trda,trdar_cd,adstrd_cd,flpop_type
0,베레베레,커피숍,2002-09-19,서울특별시 광진구 화양동 11-2번지,서울특별시 광진구 능동로13길 19,<NA>,NaN,2002-09-19,서울 광진구 능동로13길 19,37.542885,127.070832,NaN,3120053,11215710,T
1,덕수궁전통차전문,커피숍,2003-01-16,서울특별시 중구 서소문동 54-1번지 (2층),서울특별시 중구 서소문로 109,2,46.28,2003-01-22,서울 중구 서소문로 109,37.563062,126.973280,NaN,3120020,11140520,T
2,엘빠소,커피숍,2003-05-31,서울특별시 종로구 명륜2가 143-9번지,서울특별시 종로구 성균관로 18,<NA>,NaN,2003-05-31,서울 종로구 성균관로 18,37.584406,126.997641,3110021,3110021,11110650,T
3,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울특별시 종로구 세종로 82-14번지 지상1층,서울특별시 종로구 세종대로 188,1,NaN,2003-11-21,서울 종로구 세종대로 188,37.573190,126.977831,NaN,NaN,11110615,A
4,본솔카페,커피숍,2004-01-17,서울특별시 용산구 청파동2가 64-13번지 청파3가 115-12 (지상1층),서울특별시 용산구 청파로47길 71,1,13.80,2004-01-17,서울 용산구 청파로47길 71,37.544840,126.966455,3110064,3110069,11170555,T


In [106]:
# NaN 비율 확인
null_cnt = cafe_df["trdar_cd"].isna().sum()
total = len(cafe_df)
print(f"상권 미매칭: {null_cnt}/{total} ({null_cnt/total*100:.1f}%)")

상권 미매칭: 3603/14536 (24.8%)


#### 상권배후지 코드 추가

In [111]:
print(trdar_gdf.columns.tolist())

['ALLEY_TRDA', 'ALLEY_TR_1', 'XCNTS_VALU', 'YDNTS_VALU', 'SIGNGU_CD', 'SIGNGU_CD_', 'ADSTRD_CD', 'ADSTRD_CD_', 'RELM_AR', 'geometry']


In [ ]:
# # 1) cafe_df -> GeoDataFrame (실제 좌표계: WGS84)
# cafe_gdf = gpd.GeoDataFrame(
#     cafe_df.copy(),
#     geometry=gpd.points_from_xy(cafe_df["lng"], cafe_df["lat"]),
#     crs="EPSG:4326"
# )

# 2) 상권 shp 로드 (좌표계: EPSG:5181)
trdar_gdf = gpd.read_file(
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-상권배후지).zip"
)
# → 이 파일은 EPSG:5181이 이미 설정되어 있어야 함

# 3) 좌표계 통일 (cafe를 5181로 변환)
'''
폴리곤(상권)은 꼭짓점이 많아서 변환 시 오차가 누적될 수 있으므로 shp(5181) 기준으로 맞춤
'''
cafe_gdf = cafe_gdf.to_crs("EPSG:5181")  # shp 기준으로 맞춤

# 4) 공간조인
trdar_cols = ["ALLEY_TRDA", "geometry"]
joined = gpd.sjoin(
    cafe_gdf,
    trdar_gdf[trdar_cols],
    how="left",
    predicate="within"
)

# 5) 원본 인덱스 기준으로 TRDAR_CD 붙이기
cafe_df = cafe_df.copy()
cafe_df.loc[joined.index, "alley_trda"] = joined["ALLEY_TRDA"].values

#### 행정동 코드 추가

In [ ]:
# # 1) cafe_df -> GeoDataFrame (실제 좌표계: WGS84)
# cafe_gdf = gpd.GeoDataFrame(
#     cafe_df.copy(),
#     geometry=gpd.points_from_xy(cafe_df["lng"], cafe_df["lat"]),
#     crs="EPSG:4326"
# )

# 2) 상권 shp 로드 (좌표계: EPSG:5181)
trdar_gdf = gpd.read_file(
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-행정동).zip"
)
# → 이 파일은 EPSG:5181이 이미 설정되어 있어야 함

# 3) 좌표계 통일 (cafe를 5181로 변환)
'''
폴리곤(상권)은 꼭짓점이 많아서 변환 시 오차가 누적될 수 있으므로 shp(5181) 기준으로 맞춤
'''
cafe_gdf = cafe_gdf.to_crs("EPSG:5181")  # shp 기준으로 맞춤

# 4) 공간조인
trdar_cols = ["ADSTRD_CD", "geometry"]
joined = gpd.sjoin(
    cafe_gdf,
    trdar_gdf[trdar_cols],
    how="left",
    predicate="within"
)

# 5) 원본 인덱스 기준으로 adstrd_cd 붙이기
cafe_df.loc[joined.index, "adstrd_cd"] = joined["ADSTRD_CD"].values
cafe_df.head()

,nm,type,apv_date,addr,rd_addr,floor,sitearea,last_updated,kakao_addr,lat,lng,alley_trda,trdar_cd,adstrd_cd
0,베레베레,커피숍,2002-09-19,서울특별시 광진구 화양동 11-2번지,서울특별시 광진구 능동로13길 19,<NA>,NaN,2002-09-19,서울 광진구 능동로13길 19,37.542885,127.070832,NaN,3120053,11215710
2,덕수궁전통차전문,커피숍,2003-01-16,서울특별시 중구 서소문동 54-1번지 (2층),서울특별시 중구 서소문로 109,2,46.28,2003-01-22,서울 중구 서소문로 109,37.563062,126.973280,NaN,3120020,11140520
3,엘빠소,커피숍,2003-05-31,서울특별시 종로구 명륜2가 143-9번지,서울특별시 종로구 성균관로 18,<NA>,NaN,2003-05-31,서울 종로구 성균관로 18,37.584406,126.997641,3110021,3110021,11110650
4,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울특별시 종로구 세종로 82-14번지 지상1층,서울특별시 종로구 세종대로 188,1,NaN,2003-11-21,서울 종로구 세종대로 188,37.573190,126.977831,NaN,NaN,11110615
5,본솔카페,커피숍,2004-01-17,서울특별시 용산구 청파동2가 64-13번지 청파3가 115-12 (지상1층),서울특별시 용산구 청파로47길 71,1,13.80,2004-01-17,서울 용산구 청파로47길 71,37.544840,126.966455,3110064,3110069,11170555


#### 한 번에 추가하기

In [29]:
# GeoDataFrame 한 번만 생성
cafe_gdf = gpd.GeoDataFrame(
    cafe_df.copy(),
    geometry=gpd.points_from_xy(cafe_df["lng"], cafe_df["lat"]),
    crs="EPSG:4326"
)

# 각각 호출
cafe_df = spatial_join_code(cafe_df, cafe_gdf,
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-상권).zip",
    "TRDAR_CD", "trdar_cd")

cafe_df = spatial_join_code(cafe_df, cafe_gdf,
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-상권배후지).zip",
    "ALLEY_TRDA", "alley_trda")

cafe_df = spatial_join_code(cafe_df, cafe_gdf,
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-행정동).zip",
    "ADSTRD_CD", "adstrd_cd")

In [30]:
cafe_df[["trdar_cd", "alley_trda", "adstrd_cd"]].isna().sum()

trdar_cd      3601
alley_trda    4194
adstrd_cd        0
dtype: int64

#### 코드 type 추가 및 코드 값 처리

In [31]:
# 1) flpop_type 컬럼 생성
cafe_df["flpop_type"] = "A"  # 기본값
cafe_df.loc[cafe_df["alley_trda"].notna(), "flpop_type"] = "T"
cafe_df.loc[cafe_df["trdar_cd"].notna(), "flpop_type"] = "T"

# 2) trdar_cd 빈 값을 alley_trda로 대체
'''
상권-유동인구데이터, 상권배후지-유동인구데이터는 같은 API를 사용하므로 별도로 구분할 필요가 없음
'''
mask = cafe_df["trdar_cd"].isna() & cafe_df["alley_trda"].notna()
cafe_df.loc[mask, "trdar_cd"] = cafe_df.loc[mask, "alley_trda"]

### 저장 용도 dataframe 생성

In [32]:
# 1) 저장할 데이터만 추출
save_cafe_df = cafe_df[['nm', 'type', 'apv_date', 'kakao_addr', 'floor', 'sitearea',
                        'lat', 'lng', 'flpop_type', 'trdar_cd', 'adstrd_cd']]

# 2) 열이름 재변경
save_cafe_df = save_cafe_df.rename(columns={'kakao_addr': 'addr'})
save_cafe_df.head()

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,adstrd_cd
0,베레베레,커피숍,2002-09-19,서울 광진구 능동로13길 19,<NA>,NaN,37.542885,127.070832,T,3120053,11215710
2,덕수궁전통차전문,커피숍,2003-01-16,서울 중구 서소문로 109,2,46.28,37.563062,126.973280,T,3120020,11140520
3,엘빠소,커피숍,2003-05-31,서울 종로구 성균관로 18,<NA>,NaN,37.584406,126.997641,T,3110021,11110650
4,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울 종로구 세종대로 188,1,NaN,37.573190,126.977831,A,NaN,11110615
5,본솔카페,커피숍,2004-01-17,서울 용산구 청파로47길 71,1,13.80,37.544840,126.966455,T,3110069,11170555


In [33]:
# index 재설정
save_cafe_df.reset_index(drop=True, inplace=True) # index 재설정
save_cafe_df

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,adstrd_cd
0,베레베레,커피숍,2002-09-19,서울 광진구 능동로13길 19,<NA>,NaN,37.542885,127.070832,T,3120053,11215710
1,덕수궁전통차전문,커피숍,2003-01-16,서울 중구 서소문로 109,2,46.28,37.563062,126.973280,T,3120020,11140520
2,엘빠소,커피숍,2003-05-31,서울 종로구 성균관로 18,<NA>,NaN,37.584406,126.997641,T,3110021,11110650
3,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울 종로구 세종대로 188,1,NaN,37.573190,126.977831,A,NaN,11110615
4,본솔카페,커피숍,2004-01-17,서울 용산구 청파로47길 71,1,13.80,37.544840,126.966455,T,3110069,11170555
...,...,...,...,...,...,...,...,...,...,...,...
14528,카페 라희,커피숍,2026-02-27,서울 강동구 천호대로 1239,1,25.20,37.537861,127.148277,T,3111077,11740685
14529,달콤한 오늘,커피숍,2026-02-27,서울 강서구 마곡서1로 111-12,1,62.33,37.566787,126.817788,T,3110639,11500630
14530,쿠즈코커피 마포성산점,커피숍,2026-02-23,서울 마포구 월드컵북로 73,1,35.43,37.560317,126.915972,T,3110557,11440720
14531,푸디스트(주)서울시중구청카페,커피숍,2026-02-27,서울 중구 창경궁로 17,1,11.94,37.564120,126.998010,T,3120034,11140590


# 서울시 상권분석서비스(길단위인구-상권/상권배후지)
카페 데이터(cafe.csv)에서 flpop_type이 T 인 경우 사용

## 데이터 불러오기

In [148]:
apikey  = os.getenv("MJ_APIKEY")     # 인증키
start_index = "1"    # 요청시작위치
end_index   = "300"  # 요청종료위치

url = f"http://openapi.seoul.go.kr:8088/{apikey}/json/VwsmTrdarFlpopQq/{start_index}/{end_index}/"

response = requests.get(url)
print(response.content.decode('utf-8'))

{"VwsmTrdarFlpopQq":{"list_total_count":44536,"RESULT":{"CODE":"INFO-000","MESSAGE":"정상 처리되었습니다"},"row":[{"STDR_YYQU_CD":"20191","TRDAR_SE_CD":"D","TRDAR_SE_CD_NM":"발달상권","TRDAR_CD":"3120196","TRDAR_CD_NM":"서울세관","TOT_FLPOP_CO":700352.0,"ML_FLPOP_CO":359756.0,"FML_FLPOP_CO":340594.0,"AGRDE_10_FLPOP_CO":44888.0,"AGRDE_20_FLPOP_CO":140916.0,"AGRDE_30_FLPOP_CO":182543.0,"AGRDE_40_FLPOP_CO":136481.0,"AGRDE_50_FLPOP_CO":80938.0,"AGRDE_60_ABOVE_FLPOP_CO":114586.0,"TMZON_00_06_FLPOP_CO":104869.0,"TMZON_06_11_FLPOP_CO":139300.0,"TMZON_11_14_FLPOP_CO":125560.0,"TMZON_14_17_FLPOP_CO":124427.0,"TMZON_17_21_FLPOP_CO":136567.0,"TMZON_21_24_FLPOP_CO":69630.0,"MON_FLPOP_CO":105462.0,"TUES_FLPOP_CO":106420.0,"WED_FLPOP_CO":110506.0,"THUR_FLPOP_CO":115069.0,"FRI_FLPOP_CO":111466.0,"SAT_FLPOP_CO":80262.0,"SUN_FLPOP_CO":71166.0},{"STDR_YYQU_CD":"20191","TRDAR_SE_CD":"D","TRDAR_SE_CD_NM":"발달상권","TRDAR_CD":"3120197","TRDAR_CD_NM":"역삼역","TOT_FLPOP_CO":5123674.0,"ML_FLPOP_CO":2802851.0,"FML_FLPOP_CO":2320823

In [149]:
# "list_total_count":44536 ⇒ 1000개씩 나눠서 가져와야 함
apikey = os.getenv("MJ_APIKEY")
service = "VwsmTrdarFlpopQq"
base = "http://openapi.seoul.go.kr:8088"

# 1) 전체 건수 확인
first = requests.get(f"{base}/{apikey}/json/{service}/1/1/").json()
total = first[service]["list_total_count"]

# 2) 분할 수집
batch = 1000   # 서비스 허용 범위 내에서 조절
all_rows = []

for start in range(1, total + 1, batch):
    end = min(start + batch - 1, total)
    url = f"{base}/{apikey}/json/{service}/{start}/{end}/"
    response = requests.get(url).json()
    rows = response.get(service, {}).get("row", [])
    all_rows.extend(rows)

print("total:", total)
print("fetched:", len(all_rows))

total: 44536
fetched: 44536


In [150]:
flpop_tr_df = pd.DataFrame(all_rows)
flpop_tr_df.head()

,STDR_YYQU_CD,TRDAR_SE_CD,TRDAR_SE_CD_NM,TRDAR_CD,TRDAR_CD_NM,TOT_FLPOP_CO,ML_FLPOP_CO,FML_FLPOP_CO,AGRDE_10_FLPOP_CO,AGRDE_20_FLPOP_CO,...,TMZON_14_17_FLPOP_CO,TMZON_17_21_FLPOP_CO,TMZON_21_24_FLPOP_CO,MON_FLPOP_CO,TUES_FLPOP_CO,WED_FLPOP_CO,THUR_FLPOP_CO,FRI_FLPOP_CO,SAT_FLPOP_CO,SUN_FLPOP_CO
0,20191,D,발달상권,3120196,서울세관,700352.0,359756.0,340594.0,44888.0,140916.0,...,124427.0,136567.0,69630.0,105462.0,106420.0,110506.0,115069.0,111466.0,80262.0,71166.0
1,20191,D,발달상권,3120197,역삼역,5123674.0,2802851.0,2320823.0,189741.0,1248859.0,...,1029695.0,1002107.0,435697.0,824944.0,821220.0,856904.0,902709.0,858099.0,480318.0,379483.0
2,20191,D,발달상권,3120198,구역삼세무서,2735527.0,1437771.0,1297755.0,170295.0,663373.0,...,406033.0,459925.0,296022.0,414386.0,410294.0,421127.0,434270.0,419333.0,323450.0,312668.0
3,20191,D,발달상권,3120199,경복아파트교차로,505522.0,259538.0,245984.0,27105.0,109292.0,...,80078.0,87825.0,50350.0,75708.0,74958.0,77918.0,79473.0,77901.0,62424.0,57140.0
4,20191,D,발달상권,3120200,학동사거리,1349117.0,632069.0,717048.0,97382.0,326760.0,...,256961.0,289717.0,139114.0,181562.0,193744.0,203127.0,212952.0,214384.0,195706.0,147642.0


## 데이터 전처리

In [151]:
# 필요한 데이터만 추출하고, 열이름 모두 소문자로 통일
flpop_tr_df = flpop_tr_df.drop(columns=["TRDAR_SE_CD"], errors="ignore")  # 해당 열 제거
flpop_tr_df.columns = flpop_tr_df.columns.str.lower()            # flpop_df 기준으로 소문자화

flpop_tr_df

,stdr_yyqu_cd,trdar_se_cd_nm,trdar_cd,trdar_cd_nm,tot_flpop_co,ml_flpop_co,fml_flpop_co,agrde_10_flpop_co,agrde_20_flpop_co,agrde_30_flpop_co,...,tmzon_14_17_flpop_co,tmzon_17_21_flpop_co,tmzon_21_24_flpop_co,mon_flpop_co,tues_flpop_co,wed_flpop_co,thur_flpop_co,fri_flpop_co,sat_flpop_co,sun_flpop_co
0,20191,발달상권,3120196,서울세관,700352.0,359756.0,340594.0,44888.0,140916.0,182543.0,...,124427.0,136567.0,69630.0,105462.0,106420.0,110506.0,115069.0,111466.0,80262.0,71166.0
1,20191,발달상권,3120197,역삼역,5123674.0,2802851.0,2320823.0,189741.0,1248859.0,1489998.0,...,1029695.0,1002107.0,435697.0,824944.0,821220.0,856904.0,902709.0,858099.0,480318.0,379483.0
2,20191,발달상권,3120198,구역삼세무서,2735527.0,1437771.0,1297755.0,170295.0,663373.0,738048.0,...,406033.0,459925.0,296022.0,414386.0,410294.0,421127.0,434270.0,419333.0,323450.0,312668.0
3,20191,발달상권,3120199,경복아파트교차로,505522.0,259538.0,245984.0,27105.0,109292.0,147631.0,...,80078.0,87825.0,50350.0,75708.0,74958.0,77918.0,79473.0,77901.0,62424.0,57140.0
4,20191,발달상권,3120200,학동사거리,1349117.0,632069.0,717048.0,97382.0,326760.0,360568.0,...,256961.0,289717.0,139114.0,181562.0,193744.0,203127.0,212952.0,214384.0,195706.0,147642.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44531,20252,관광특구,3001492,명동 남대문 북창동 다동 무교동 관광특구,6952851.0,3278243.0,3674608.0,287579.0,1200664.0,1661817.0,...,1702240.0,1300602.0,344642.0,1096780.0,1095901.0,1183241.0,1155247.0,1105066.0,721777.0,594838.0
44532,20252,관광특구,3001493,동대문패션타운 관광특구,3489708.0,1624950.0,1864757.0,196001.0,649607.0,775338.0,...,551648.0,660777.0,416430.0,536087.0,541289.0,557366.0,555502.0,516952.0,392095.0,390417.0
44533,20252,관광특구,3001494,종로·청계 관광특구,8537311.0,4476428.0,4060884.0,322168.0,1533817.0,1665463.0,...,1786024.0,1686973.0,684131.0,1318020.0,1311528.0,1399371.0,1376796.0,1329573.0,1005568.0,796454.0
44534,20252,관광특구,3001495,잠실 관광특구,4116967.0,1951388.0,2165578.0,407384.0,949932.0,1026832.0,...,658382.0,900623.0,483598.0,565714.0,570891.0,574479.0,589440.0,616438.0,619744.0,580259.0


In [159]:
flpop_tr_df['stdr_yyqu_cd'].unique()

<StringArray>
['20191', '20241', '20214', '20211', '20222', '20231', '20234', '20203',
 '20232', '20252', '20221', '20213', '20192', '20242', '20251', '20201',
 '20204', '20243', '20233', '20193', '20223', '20212', '20244', '20194',
 '20202', '20224', '20253']
Length: 27, dtype: str

In [160]:
flpop_tr_df = (
    flpop_tr_df
    .sort_values(["stdr_yyqu_cd", "trdar_cd"], ascending=[True, True])
    .reset_index(drop=True)
)
flpop_tr_df

,stdr_yyqu_cd,trdar_se_cd_nm,trdar_cd,trdar_cd_nm,tot_flpop_co,ml_flpop_co,fml_flpop_co,agrde_10_flpop_co,agrde_20_flpop_co,agrde_30_flpop_co,...,tmzon_14_17_flpop_co,tmzon_17_21_flpop_co,tmzon_21_24_flpop_co,mon_flpop_co,tues_flpop_co,wed_flpop_co,thur_flpop_co,fri_flpop_co,sat_flpop_co,sun_flpop_co
0,20191,관광특구,3001491,이태원 관광특구,2418524.0,1227335.0,1191188.0,142554.0,722271.0,562230.0,...,338263.0,514860.0,349263.0,301439.0,304054.0,306360.0,317331.0,363506.0,438229.0,387605.0
1,20191,관광특구,3001492,명동 남대문 북창동 다동 무교동 관광특구,7851811.0,3804599.0,4047212.0,295027.0,1475672.0,1880623.0,...,1997410.0,1573937.0,411649.0,1218293.0,1204925.0,1259889.0,1340506.0,1317245.0,875006.0,635949.0
2,20191,관광특구,3001493,동대문패션타운 관광특구,3779788.0,1827835.0,1951954.0,188911.0,747125.0,880198.0,...,589953.0,672029.0,465172.0,560580.0,556120.0,571025.0,595428.0,601168.0,501537.0,393931.0
3,20191,관광특구,3001494,종로·청계 관광특구,10936954.0,5904661.0,5032294.0,447361.0,2032682.0,1943111.0,...,2437343.0,2319158.0,916464.0,1636435.0,1633361.0,1688397.0,1783350.0,1802017.0,1425047.0,968348.0
4,20191,관광특구,3001495,잠실 관광특구,3853206.0,1842185.0,2011021.0,394695.0,776760.0,830510.0,...,614804.0,800989.0,448744.0,527554.0,540737.0,550151.0,561649.0,570399.0,574518.0,528198.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44531,20253,전통시장,3130323,둔촌역전통시장,301735.0,142627.0,159109.0,38870.0,35722.0,41641.0,...,32695.0,48635.0,39718.0,43350.0,42974.0,42971.0,42651.0,43291.0,43850.0,42648.0
44532,20253,전통시장,3130324,길동복조리시장,533011.0,235646.0,297366.0,73020.0,57427.0,73982.0,...,53013.0,80821.0,76017.0,75837.0,76541.0,76326.0,75458.0,76812.0,75707.0,76330.0
44533,20253,전통시장,3130325,명일전통시장,294819.0,131330.0,163490.0,52824.0,31347.0,37791.0,...,40989.0,62090.0,37379.0,41220.0,40976.0,42073.0,40997.0,41958.0,42827.0,44770.0
44534,20253,전통시장,3130326,고덕 골목형상점가,88556.0,39833.0,48725.0,16548.0,7576.0,14690.0,...,10922.0,16408.0,11142.0,12458.0,12520.0,12663.0,12545.0,12667.0,13037.0,12667.0


# 서울시 상권분석서비스(길단위인구-행정동)
카페 데이터(cafe.csv)에서 flpop_type이 A 인 경우 사용

## 데이터 불러오기

In [163]:
apikey  = os.getenv("MJ_APIKEY")     # 인증키
start_index = "1"    # 요청시작위치
end_index   = "300"  # 요청종료위치

url = f"http://openapi.seoul.go.kr:8088/{apikey}/json/VwsmAdstrdFlpopW/{start_index}/{end_index}/"

response = requests.get(url)
print(response.content.decode('utf-8'))

{"VwsmAdstrdFlpopW":{"list_total_count":11475,"RESULT":{"CODE":"INFO-000","MESSAGE":"정상 처리되었습니다"},"row":[{"STDR_YYQU_CD":"20241","ADSTRD_CD":"11320515","ADSTRD_CD_NM":"창5동","TOT_FLPOP_CO":4263862.0,"ML_FLPOP_CO":1929777.0,"FML_FLPOP_CO":2334085.0,"AGRDE_10_FLPOP_CO":678202.0,"AGRDE_20_FLPOP_CO":541171.0,"AGRDE_30_FLPOP_CO":608372.0,"AGRDE_40_FLPOP_CO":616520.0,"AGRDE_50_FLPOP_CO":695087.0,"AGRDE_60_ABOVE_FLPOP_CO":1124511.0,"TMZON_00_06_FLPOP_CO":954982.0,"TMZON_06_11_FLPOP_CO":861032.0,"TMZON_11_14_FLPOP_CO":538239.0,"TMZON_14_17_FLPOP_CO":558731.0,"TMZON_17_21_FLPOP_CO":806202.0,"TMZON_21_24_FLPOP_CO":544678.0,"MON_FLPOP_CO":602886.0,"TUES_FLPOP_CO":597483.0,"WED_FLPOP_CO":596701.0,"THUR_FLPOP_CO":597592.0,"FRI_FLPOP_CO":602498.0,"SAT_FLPOP_CO":634238.0,"SUN_FLPOP_CO":632462.0},{"STDR_YYQU_CD":"20241","ADSTRD_CD":"11320521","ADSTRD_CD_NM":"도봉1동","TOT_FLPOP_CO":3625772.0,"ML_FLPOP_CO":1774866.0,"FML_FLPOP_CO":1850906.0,"AGRDE_10_FLPOP_CO":407131.0,"AGRDE_20_FLPOP_CO":355605.0,"AGRDE_3

In [164]:
# "list_total_count":11475 ⇒ 1000개씩 나눠서 가져와야 함
apikey = os.getenv("MJ_APIKEY")
service = "VwsmAdstrdFlpopW"
base = "http://openapi.seoul.go.kr:8088"

# 1) 전체 건수 확인
first = requests.get(f"{base}/{apikey}/json/{service}/1/1/").json()
total = first[service]["list_total_count"]

# 2) 분할 수집
batch = 1000   # 서비스 허용 범위 내에서 조절
all_rows = []

for start in range(1, total + 1, batch):
    end = min(start + batch - 1, total)
    url = f"{base}/{apikey}/json/{service}/{start}/{end}/"
    response = requests.get(url).json()
    rows = response.get(service, {}).get("row", [])
    all_rows.extend(rows)

print("total:", total)
print("fetched:", len(all_rows))

total: 11475
fetched: 11475


In [165]:
flpop_ad_df = pd.DataFrame(all_rows)
flpop_ad_df.head()

,STDR_YYQU_CD,ADSTRD_CD,ADSTRD_CD_NM,TOT_FLPOP_CO,ML_FLPOP_CO,FML_FLPOP_CO,AGRDE_10_FLPOP_CO,AGRDE_20_FLPOP_CO,AGRDE_30_FLPOP_CO,AGRDE_40_FLPOP_CO,...,TMZON_14_17_FLPOP_CO,TMZON_17_21_FLPOP_CO,TMZON_21_24_FLPOP_CO,MON_FLPOP_CO,TUES_FLPOP_CO,WED_FLPOP_CO,THUR_FLPOP_CO,FRI_FLPOP_CO,SAT_FLPOP_CO,SUN_FLPOP_CO
0,20241,11320515,창5동,4263862.0,1929777.0,2334085.0,678202.0,541171.0,608372.0,616520.0,...,558731.0,806202.0,544678.0,602886.0,597483.0,596701.0,597592.0,602498.0,634238.0,632462.0
1,20241,11320521,도봉1동,3625772.0,1774866.0,1850906.0,407131.0,355605.0,408412.0,484567.0,...,430102.0,602094.0,467208.0,507631.0,505812.0,498030.0,504924.0,508073.0,541035.0,560267.0
2,20241,11320522,도봉2동,3404141.0,1613793.0,1790348.0,506139.0,343526.0,441671.0,491399.0,...,377885.0,552874.0,456093.0,488117.0,484012.0,479359.0,481832.0,481692.0,490190.0,498941.0
3,20241,11320660,쌍문1동,5148281.0,2292513.0,2855768.0,894616.0,716270.0,614891.0,722306.0,...,562128.0,796422.0,680556.0,737177.0,730866.0,721639.0,726841.0,717698.0,741625.0,772434.0
4,20241,11320670,쌍문2동,4539787.0,2089232.0,2450556.0,713920.0,536325.0,632059.0,653006.0,...,519171.0,745124.0,598256.0,648956.0,640168.0,638320.0,641386.0,639705.0,658679.0,672574.0


## 데이터 전처리

In [166]:
# 열이름 모두 소문자로 통일
flpop_ad_df.columns = flpop_ad_df.columns.str.lower() 

# 정렬
flpop_ad_df = (
    flpop_ad_df
    .sort_values(["stdr_yyqu_cd", "adstrd_cd"], ascending=[True, True])
    .reset_index(drop=True)
)

flpop_ad_df.head()

,stdr_yyqu_cd,adstrd_cd,adstrd_cd_nm,tot_flpop_co,ml_flpop_co,fml_flpop_co,agrde_10_flpop_co,agrde_20_flpop_co,agrde_30_flpop_co,agrde_40_flpop_co,...,tmzon_14_17_flpop_co,tmzon_17_21_flpop_co,tmzon_21_24_flpop_co,mon_flpop_co,tues_flpop_co,wed_flpop_co,thur_flpop_co,fri_flpop_co,sat_flpop_co,sun_flpop_co
0,20191,11110515,청운효자동,3668225.0,1693148.0,1975079.0,604553.0,542715.0,581784.0,682867.0,...,460874.0,558750.0,434296.0,528566.0,528092.0,533157.0,540482.0,534862.0,505661.0,497404.0
1,20191,11110530,사직동,4628956.0,2290446.0,2338512.0,403260.0,752637.0,920942.0,904937.0,...,820984.0,887499.0,442738.0,682476.0,685164.0,709260.0,729241.0,717359.0,570596.0,534862.0
2,20191,11110540,삼청동,1010831.0,486563.0,524268.0,97480.0,152090.0,189542.0,182516.0,...,186331.0,173031.0,87952.0,142143.0,144846.0,146730.0,150008.0,152319.0,144807.0,129977.0
3,20191,11110550,부암동,1053668.0,468695.0,584973.0,169130.0,143980.0,124425.0,159491.0,...,125124.0,164822.0,132619.0,152370.0,152256.0,152123.0,150560.0,147122.0,148571.0,150668.0
4,20191,11110560,평창동,1139144.0,497078.0,642067.0,188796.0,119637.0,120384.0,165176.0,...,127608.0,173849.0,148127.0,161708.0,162637.0,161310.0,161151.0,159959.0,164827.0,167552.0


# 직방(매물) 데이터에 유동인구 데이터 코드 추가하기

In [191]:
# 데이터 불러오기
listing_df = pd.read_csv(r"C:\vs_code_prj\brewmap\data\mj\test_zigbang.csv")
listing_df.head()

,listing_id,listing_status,address,lat,lng,business_type,transaction_type,sale_price,key_money,deposit,monthly_rent,maintenance_fee,size_m2,floor
0,575115,open,서울특별시 강남구 논현동,37.519306,127.029262,사무실,월세,0,NaN,30000000,2000000,0,82.60,4층
1,703777,open,서울특별시 서초구 잠원동,37.511775,127.020540,카페/커피,월세,0,40000000.0,50000000,3000000,0,33.06,1층
2,701846,open,서울특별시 강남구 논현동,37.512658,127.021577,사무실,월세,0,NaN,35000000,3000000,0,49.59,2층
3,455221,open,서울특별시 강남구 논현동,37.509032,127.023453,한식/고기집,월세,0,220000000.0,80000000,5500000,1200000,264.50,2층
4,394623,open,서울특별시 강남구 논현동,37.515856,127.030844,가구/가전,월세,0,200000000.0,50000000,3100000,600000,165.00,1층


In [192]:
# 코드 추가
# GeoDataFrame 한 번만 생성
listing_gdf = gpd.GeoDataFrame(
    listing_df.copy(),
    geometry=gpd.points_from_xy(listing_df["lng"], listing_df["lat"]),
    crs="EPSG:4326"
)

# 각각 호출
listing_df = spatial_join_code(listing_df, listing_gdf,
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-상권).zip",
    "TRDAR_CD", "trdar_cd")

listing_df = spatial_join_code(listing_df, listing_gdf,
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-상권배후지).zip",
    "ALLEY_TRDA", "alley_trda")

listing_df = spatial_join_code(listing_df, listing_gdf,
    r"C:\vs_code_prj\brewmap\data\mj\서울시 상권분석서비스(영역-행정동).zip",
    "ADSTRD_CD", "adstrd_cd")

listing_df.head()

,listing_id,listing_status,address,lat,lng,business_type,transaction_type,sale_price,key_money,deposit,monthly_rent,maintenance_fee,size_m2,floor,trdar_cd,alley_trda,adstrd_cd
0,575115,open,서울특별시 강남구 논현동,37.519306,127.029262,사무실,월세,0,NaN,30000000,2000000,0,82.60,4층,3120190,3110957,11680531
1,703777,open,서울특별시 서초구 잠원동,37.511775,127.020540,카페/커피,월세,0,40000000.0,50000000,3000000,0,33.06,1층,3120185,3110950,11650540
2,701846,open,서울특별시 강남구 논현동,37.512658,127.021577,사무실,월세,0,NaN,35000000,3000000,0,49.59,2층,3120185,3110950,11680521
3,455221,open,서울특별시 강남구 논현동,37.509032,127.023453,한식/고기집,월세,0,220000000.0,80000000,5500000,1200000,264.50,2층,3130303,NaN,11680521
4,394623,open,서울특별시 강남구 논현동,37.515856,127.030844,가구/가전,월세,0,200000000.0,50000000,3100000,600000,165.00,1층,3120191,3110957,11680531


#### 코드 type 추가 및 코드 값 처리

In [193]:
# 1) flpop_type 컬럼 생성
listing_df["flpop_type"] = "A"  # 기본값
listing_df.loc[listing_df["alley_trda"].notna(), "flpop_type"] = "T"
listing_df.loc[listing_df["trdar_cd"].notna(), "flpop_type"] = "T"

# 2) trdar_cd 빈 값을 alley_trda로 대체
'''
상권-유동인구데이터, 상권배후지-유동인구데이터는 같은 API를 사용하므로 별도로 구분할 필요가 없음
'''
mask = listing_df["trdar_cd"].isna() & listing_df["alley_trda"].notna()
listing_df.loc[mask, "trdar_cd"] = listing_df.loc[mask, "alley_trda"]
listing_df.head()

,listing_id,listing_status,address,lat,lng,business_type,transaction_type,sale_price,key_money,deposit,monthly_rent,maintenance_fee,size_m2,floor,trdar_cd,alley_trda,adstrd_cd,flpop_type
0,575115,open,서울특별시 강남구 논현동,37.519306,127.029262,사무실,월세,0,NaN,30000000,2000000,0,82.60,4층,3120190,3110957,11680531,T
1,703777,open,서울특별시 서초구 잠원동,37.511775,127.020540,카페/커피,월세,0,40000000.0,50000000,3000000,0,33.06,1층,3120185,3110950,11650540,T
2,701846,open,서울특별시 강남구 논현동,37.512658,127.021577,사무실,월세,0,NaN,35000000,3000000,0,49.59,2층,3120185,3110950,11680521,T
3,455221,open,서울특별시 강남구 논현동,37.509032,127.023453,한식/고기집,월세,0,220000000.0,80000000,5500000,1200000,264.50,2층,3130303,NaN,11680521,T
4,394623,open,서울특별시 강남구 논현동,37.515856,127.030844,가구/가전,월세,0,200000000.0,50000000,3100000,600000,165.00,1층,3120191,3110957,11680531,T


## 저장용 dataframe 생성

In [ ]:
# 저장 전 결측치 확인 → key_money 전처리 필요
listing_df.isna().sum()

listing_id            0
listing_status        0
address               0
lat                   0
lng                   0
business_type         0
transaction_type      0
sale_price            0
key_money           465
deposit               0
monthly_rent          0
maintenance_fee       0
size_m2               0
floor                 0
trdar_cd              0
alley_trda          223
adstrd_cd             0
flpop_type            0
dtype: int64

### 데이터 전처리

In [194]:
listing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1319 entries, 0 to 1318
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   listing_id        1319 non-null   int64  
 1   listing_status    1319 non-null   str    
 2   address           1319 non-null   str    
 3   lat               1319 non-null   float64
 4   lng               1319 non-null   float64
 5   business_type     1319 non-null   str    
 6   transaction_type  1319 non-null   str    
 7   sale_price        1319 non-null   int64  
 8   key_money         854 non-null    float64
 9   deposit           1319 non-null   int64  
 10  monthly_rent      1319 non-null   int64  
 11  maintenance_fee   1319 non-null   int64  
 12  size_m2           1319 non-null   float64
 13  floor             1319 non-null   str    
 14  trdar_cd          1319 non-null   str    
 15  alley_trda        1096 non-null   str    
 16  adstrd_cd         1319 non-null   str    
 17  flpop_

In [199]:
# key_money 타입 변경 및 결측치 0으로 변경
listing_df["key_money"] = listing_df["key_money"].fillna(0).astype(int)

In [200]:
listing_df["key_money"].isna().sum(), listing_df["key_money"].dtype

(np.int64(0), dtype('int64'))

In [205]:
print(listing_df['listing_status'].unique())
print(listing_df['transaction_type'].unique())

<StringArray>
['open']
Length: 1, dtype: str
<StringArray>
['월세', '매매', '전세']
Length: 3, dtype: str


### 필요한 데이터만 추출

In [204]:
save_listing_df = (
    listing_df
    .drop(columns=["listing_status", "alley_trda"])
    .rename(columns={"address": "addr"})
)

# 마지막 컬럼 순서 지정
front_cols = [c for c in save_listing_df.columns if c not in ["flpop_type", "trdar_cd", "adstrd_cd"]]
save_listing_df = save_listing_df[front_cols + ["flpop_type", "trdar_cd", "adstrd_cd"]]

save_listing_df.head()

,listing_id,addr,lat,lng,business_type,transaction_type,sale_price,key_money,deposit,monthly_rent,maintenance_fee,size_m2,floor,flpop_type,trdar_cd,adstrd_cd
0,575115,서울특별시 강남구 논현동,37.519306,127.029262,사무실,월세,0,0,30000000,2000000,0,82.60,4층,T,3120190,11680531
1,703777,서울특별시 서초구 잠원동,37.511775,127.020540,카페/커피,월세,0,40000000,50000000,3000000,0,33.06,1층,T,3120185,11650540
2,701846,서울특별시 강남구 논현동,37.512658,127.021577,사무실,월세,0,0,35000000,3000000,0,49.59,2층,T,3120185,11680521
3,455221,서울특별시 강남구 논현동,37.509032,127.023453,한식/고기집,월세,0,220000000,80000000,5500000,1200000,264.50,2층,T,3130303,11680521
4,394623,서울특별시 강남구 논현동,37.515856,127.030844,가구/가전,월세,0,200000000,50000000,3100000,600000,165.00,1층,T,3120191,11680531


# csv로 내보내기

In [147]:
save_center_df.to_csv("industry_center.csv", index=False, encoding="utf-8-sig")

In [34]:
save_cafe_df.to_csv("cafe.csv", index=False, encoding="utf-8-sig")

In [168]:
flpop_tr_df.to_csv("flpop_t.csv", index=False, encoding="utf-8-sig")
flpop_ad_df.to_csv("flpop_a.csv", index=False, encoding="utf-8-sig")

In [206]:
save_listing_df.to_csv("listing.csv", index=False, encoding="utf-8-sig")